# 🚀 PyTorch GPU-Optimized Reactive LightGBM vs LSTM

## Comparison Update
✅ **Reactive LightGBM**: Upgraded from ARIMA to Gradient Boosting with short-term lags (1h, 2h, 3h) to match LSTM's information access.
✅ **PyTorch LSTM**: Deep Learning model with 60h lookback.

## Key Features:
- PyTorch LSTM with CUDA acceleration
- Automatic mixed precision (AMP) training
- Efficient data loading with DataLoader
- **New**: Reactive LightGBM implementation
- Real-time GPU monitoring

## 1. Setup and GPU Configuration

In [ ]:
# =============================================================================
# PYTORCH GPU SETUP AND VERIFICATION
# =============================================================================

import os
import gc
import warnings
import time
import json
warnings.filterwarnings('ignore')

# Import standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Import PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler

print("🔥 PyTorch GPU Configuration")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
        print(f"  Compute Capability: {torch.cuda.get_device_properties(i).major}.{torch.cuda.get_device_properties(i).minor}")
    
    # Set default device
    device = torch.device('cuda:0')
    torch.cuda.set_device(device)
    
    # Enable cuDNN optimization
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    
    print(f"\n✅ Using GPU: {device}")
    print("✅ cuDNN optimizations enabled")
else:
    device = torch.device('cpu')
    print("\n⚠️ No GPU detected. Using CPU.")
    print("To use GPU with PyTorch:")
    print("  conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia")
    print("  or")
    print("  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

## 2. GPU Memory Management Functions

In [ ]:
# =============================================================================
# GPU MEMORY MANAGEMENT UTILITIES
# =============================================================================

def print_gpu_memory():
    """Print current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: Allocated={allocated:.2f}GB, Reserved={reserved:.2f}GB")

def clear_gpu_memory():
    """Clear GPU memory cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        print("✅ GPU memory cleared")

def move_to_device(data, device):
    """Safely move data to device"""
    if isinstance(data, (list, tuple)):
        return [move_to_device(x, device) for x in data]
    elif isinstance(data, dict):
        return {k: move_to_device(v, device) for k, v in data.items()}
    elif isinstance(data, torch.Tensor):
        return data.to(device, non_blocking=True)
    else:
        return data

print("✅ Memory management functions loaded")
print_gpu_memory()

## 3. Data Loading and Preprocessing

In [ ]:
# =============================================================================
# EFFICIENT DATA LOADING
# =============================================================================

def load_and_preprocess_data(file_path, target_col):
    """Load and preprocess time series data"""
    print("📊 Loading data...")
    
    # Try multiple file paths
    paths_to_try = [file_path, f"DataFolder/{file_path}", file_path.replace('DataFolder/', '')]
    
    df = None
    for path in paths_to_try:
        try:
            df = pd.read_csv(path, parse_dates=['Date and Time'], index_col='Date and Time')
            print(f"   ✅ Loaded from: {path}")
            break
        except:
            continue
    
    if df is None:
        raise FileNotFoundError(f"Could not find data file: {file_path}")
    
    # Extract target series
    series = df[target_col].copy().astype(np.float32)
    
    # Clear dataframe from memory
    del df
    gc.collect()
    
    # Handle missing values
    series = series.interpolate(method='linear', limit=24).ffill().bfill()
    
    # Ensure hourly frequency
    series = series.sort_index()
    full_range = pd.date_range(start=series.index.min(), end=series.index.max(), freq='h')
    series = series.reindex(full_range).interpolate(method='linear').ffill().bfill()
    
    print(f"   Shape: {len(series):,} samples")
    print(f"   Range: {series.index.min()} to {series.index.max()}")
    print(f"   Memory: {series.memory_usage(deep=True) / 1024**2:.2f} MB")
    
    return series

# Load the data
target_col = 'Hourly Demand Met (in MW)'
DATA_DIR = os.environ.get("DATA_DIR", "../data")
series = load_and_preprocess_data(os.path.join(DATA_DIR, 'final_4year.csv'), target_col)

# Train/Test split
split_ratio = 0.90
split_idx = int(len(series) * split_ratio)

train = series.iloc[:split_idx]
test = series.iloc[split_idx:]

print(f"\n📊 Data Split:")
print(f"   Train: {len(train):,} samples ({len(train)/len(series)*100:.1f}%)")
print(f"   Test:  {len(test):,} samples ({len(test)/len(series)*100:.1f}%)")

## 4. Reactive LightGBM (Gradient Boosting)

In [ ]:
# =============================================================================
# REACTIVE LIGHTGBM IMPLEMENTATION (Fair Comparison)
# =============================================================================

import lightgbm as lgb
import numpy as np
import pandas as pd
import gc

print("🔍 Fitting Reactive LightGBM (Gradient Boosting) Model...\n")

# 1. FEATURE ENGINEERING
# -----------------------------------------------------------------------------
def create_features(df, target_col):
    """
    Creates time-series features including SHORT-TERM lags for reactivity.
    """
    data = df.copy()
    
    # 1. Time-based features
    data['hour'] = data.index.hour
    data['dayofweek'] = data.index.dayofweek
    data['month'] = data.index.month
    data['is_weekend'] = (data.index.dayofweek >= 5).astype(int)
    
    # 2. Cyclical encoding
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
    data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
    
    # 3. Lag Features (The "Reactive" components)
    # CRITICAL: We add lags 1, 2, 3 to match LSTM's access to recent history
    data['lag_1'] = data[target_col].shift(1)
    data['lag_2'] = data[target_col].shift(2)
    data['lag_3'] = data[target_col].shift(3)
    
    # Seasonal lags
    data['lag_24'] = data[target_col].shift(24)
    data['lag_168'] = data[target_col].shift(168)
    
    # 4. Rolling Statistics
    data['roll_mean_6'] = data[target_col].rolling(window=6).mean().shift(1)
    data['roll_std_24'] = data[target_col].rolling(window=24).std().shift(24)
    
    return data

print("   ⏳ Engineering features (Short-term lags + Seasonality)...")

# Convert Series to DataFrame for feature engineering
df_full = series.to_frame(name='target')

# Create features on the FULL dataset
df_features = create_features(df_full, 'target')

# Drop NaNs created by lagging (first 168 hours will be lost)
df_features = df_features.dropna()

# 2. TRAIN/TEST SPLIT
# -----------------------------------------------------------------------------
# Re-split based on indices
train_size = int(len(df_features) * 0.90)

X = df_features.drop('target', axis=1)
y = df_features['target']

X_train_ml = X.iloc[:train_size]
y_train_ml = y.iloc[:train_size]
X_test_ml = X.iloc[train_size:]
y_test_ml = y.iloc[train_size:]

print(f"   Training Data: {X_train_ml.shape}")
print(f"   Testing Data:  {X_test_ml.shape}")

# 3. MODEL TRAINING
# -----------------------------------------------------------------------------
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 40,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'verbose': -1,
    'n_jobs': -1
}

try:
    print("   🚀 Training LightGBM...")
    
    lgbm_model = lgb.LGBMRegressor(**params)
    
    lgbm_model.fit(
        X_train_ml, y_train_ml,
        eval_set=[(X_test_ml, y_test_ml)],
        eval_metric='rmse',
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )
    
    print(f"   ✅ LightGBM fitted successfully")
    print(f"   Best Iteration: {lgbm_model.best_iteration_}")
    
    # Generate predictions
    lgbm_predictions = lgbm_model.predict(X_test_ml)
    
except Exception as e:
    print(f"   ⚠️ LightGBM failed: {str(e)}")
    lgbm_predictions = np.full(len(y_test_ml), y_train_ml.iloc[-1])

# 4. ALIGNMENT
# -----------------------------------------------------------------------------
lgbm_pred_series = pd.Series(lgbm_predictions, index=X_test_ml.index)
lgbm_pred_series = lgbm_pred_series.reindex(test.index)

if lgbm_pred_series.isnull().any():
    lgbm_pred_series = lgbm_pred_series.ffill().bfill()

# Assign to variable expected by downstream cells
arima_pred_series = lgbm_pred_series 

print(f"\n   Generated {len(lgbm_pred_series)} Reactive LightGBM predictions")
print(f"   Range: [{lgbm_pred_series.min():.2f}, {lgbm_pred_series.max():.2f}]")

# Cleanup
del df_full, df_features, X_train_ml, X_test_ml, lgbm_model
gc.collect()

## 5. PyTorch LSTM Data Preparation

In [ ]:
# =============================================================================
# PREPARE DATA FOR PYTORCH LSTM
# =============================================================================

# Scale the data
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train.values.reshape(-1, 1)).astype(np.float32)
test_scaled = scaler.transform(test.values.reshape(-1, 1)).astype(np.float32)

print("📊 Preparing LSTM sequences...")

# Hyperparameters
LOOKBACK = 60  # Past hours to use for prediction
BATCH_SIZE = 256 if torch.cuda.is_available() else 64
LEARNING_RATE = 0.001
EPOCHS = 30

def create_sequences(data, lookback):
    """Create sequences for LSTM training"""
    sequences = []
    targets = []
    
    for i in range(lookback, len(data)):
        sequences.append(data[i-lookback:i])
        targets.append(data[i])
    
    return np.array(sequences), np.array(targets)

# Create training sequences
X_train, y_train = create_sequences(train_scaled, LOOKBACK)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)

# Create DataLoader for batch processing
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=0,  # Set to 0 for Windows, 2-4 for Linux
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

# Validation split
val_size = int(0.1 * len(X_train_tensor))
val_dataset = TensorDataset(
    X_train_tensor[-val_size:],
    y_train_tensor[-val_size:]
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"   Training sequences: {X_train.shape}")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Batch size: {BATCH_SIZE}")

## 6. PyTorch LSTM Model Architecture

In [ ]:
# =============================================================================
# PYTORCH LSTM MODEL
# =============================================================================

class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1, dropout=0.2):
        super(LSTMModel, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        
        # Fully connected layers
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size // 2, output_size)
        
    def forward(self, x):
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0, c0))
        
        # Get last time step output
        out = out[:, -1, :]
        
        # Pass through fully connected layers
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        
        return out

# Create model
model = LSTMModel(
    input_size=1,
    hidden_size=64 if torch.cuda.is_available() else 32,
    num_layers=2,
    output_size=1,
    dropout=0.2
)

# Move model to device
model = model.to(device)

# Print model summary
print("🏗️ LSTM Model Architecture:")
print(f"   Device: {device}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Model structure
print("\nModel Structure:")
print(model)

print_gpu_memory()

## 7. Training Functions

In [ ]:
# =============================================================================
# TRAINING UTILITIES
# =============================================================================

def train_epoch(model, data_loader, criterion, optimizer, scaler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    for batch_idx, (data, targets) in enumerate(data_loader):
        # Move to device
        data = data.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Mixed precision training
        with autocast(enabled=torch.cuda.is_available()):
            outputs = model(data)
            loss = criterion(outputs, targets)
        
        # Backward pass
        if torch.cuda.is_available():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(data_loader)

def validate_epoch(model, data_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for data, targets in data_loader:
            data = data.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            
            with autocast(enabled=torch.cuda.is_available()):
                outputs = model(data)
                loss = criterion(outputs, targets)
            
            total_loss += loss.item()
    
    return total_loss / len(data_loader)

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
        return self.early_stop

print("✅ Training functions loaded")

## 8. Train the LSTM Model

In [ ]:
# =============================================================================
# TRAIN LSTM MODEL
# =============================================================================

print("🚀 Training LSTM on", "GPU" if torch.cuda.is_available() else "CPU")
print("="*60)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

# Mixed precision scaler for GPU
scaler = GradScaler(enabled=torch.cuda.is_available())

# Early stopping
early_stopping = EarlyStopping(patience=5)

# Training history
train_losses = []
val_losses = []

# Training loop
best_val_loss = float('inf')
best_model_state = None

start_time = time.time()

for epoch in range(EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
    train_losses.append(train_loss)
    
    # Validate
    val_loss = validate_epoch(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        print_gpu_memory()
    
    # Early stopping
    if early_stopping(val_loss):
        print(f"\n⚠️ Early stopping at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(best_model_state)

elapsed_time = time.time() - start_time
print(f"\n✅ Training complete!")
print(f"   Time: {elapsed_time:.1f} seconds")
print(f"   Best validation loss: {best_val_loss:.6f}")
print(f"   Final epoch: {len(train_losses)}")

## 9. Generate LSTM Predictions

In [ ]:
# =============================================================================
# GENERATE LSTM PREDICTIONS
# =============================================================================

print("📈 Generating LSTM predictions...")

model.eval()

# Combine scaled data
full_scaled = np.concatenate([train_scaled, test_scaled])
test_start_idx = len(train_scaled)

# Generate predictions
lstm_predictions_scaled = []

with torch.no_grad():
    # Process in batches for efficiency
    batch_size = 512
    test_sequences = []
    
    # Create all test sequences
    for i in range(test_start_idx, len(full_scaled)):
        seq = full_scaled[i - LOOKBACK:i]
        test_sequences.append(seq)
    
    test_sequences = np.array(test_sequences)
    
    # Predict in batches
    for i in range(0, len(test_sequences), batch_size):
        batch = test_sequences[i:i+batch_size]
        batch_tensor = torch.FloatTensor(batch).to(device)
        
        with autocast(enabled=torch.cuda.is_available()):
            batch_pred = model(batch_tensor)
        
        lstm_predictions_scaled.extend(batch_pred.cpu().numpy().flatten())
        
        # Progress update
        if i % 1000 == 0:
            print(f"   Processed {min(i+batch_size, len(test_sequences))}/{len(test_sequences)} sequences")

# Convert to numpy array
lstm_predictions_scaled = np.array(lstm_predictions_scaled).reshape(-1, 1)

# Inverse transform
lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled).flatten()
lstm_pred_series = pd.Series(lstm_predictions, index=test.index)

print(f"\n✅ Generated {len(lstm_pred_series)} predictions")
print(f"   Range: [{lstm_pred_series.min():.2f}, {lstm_pred_series.max():.2f}]")

# Clear GPU memory
clear_gpu_memory()

## 10. Model Evaluation

In [ ]:
# =============================================================================
# EVALUATION METRICS
# =============================================================================

def calculate_metrics(y_true, y_pred, model_name):
    """Calculate performance metrics"""
    y_true = np.array(y_true, dtype=np.float32)
    y_pred = np.array(y_pred, dtype=np.float32)
    
    # Remove NaN values
    valid_mask = ~(np.isnan(y_true) | np.isnan(y_pred) | np.isinf(y_true) | np.isinf(y_pred))
    
    if valid_mask.sum() == 0:
        return {
            'Model': model_name,
            'RMSE (MW)': np.nan,
            'MAE (MW)': np.nan,
            'MAPE (%)': np.nan,
            'Valid Samples': '0/0'
        }
    
    y_true_clean = y_true[valid_mask]
    y_pred_clean = y_pred[valid_mask]
    
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    
    # MAPE
    non_zero_mask = y_true_clean != 0
    if non_zero_mask.sum() > 0:
        mape = np.mean(np.abs((y_true_clean[non_zero_mask] - y_pred_clean[non_zero_mask]) / 
                              y_true_clean[non_zero_mask])) * 100
    else:
        mape = np.nan
    
    return {
        'Model': model_name,
        'RMSE (MW)': round(rmse, 2),
        'MAE (MW)': round(mae, 2),
        'MAPE (%)': round(mape, 2) if not np.isnan(mape) else np.nan,
        'Valid Samples': f"{valid_mask.sum()}/{len(y_true)}"
    }

# Calculate metrics
print("📊 Calculating metrics...\n")

arima_metrics = calculate_metrics(test.values, arima_pred_series.values, 'ARIMA')
lstm_metrics = calculate_metrics(test.values, lstm_pred_series.values, 'PyTorch LSTM')

# Create results dataframe
results_df = pd.DataFrame([arima_metrics, lstm_metrics])

print("="*70)
print("MODEL COMPARISON RESULTS")
print("="*70)
print(f"\n{results_df.to_string(index=False)}")

# Determine winner
if not np.isnan(arima_metrics['RMSE (MW)']) and not np.isnan(lstm_metrics['RMSE (MW)']):
    if arima_metrics['RMSE (MW)'] < lstm_metrics['RMSE (MW)']:
        winner = 'ARIMA'
        margin = lstm_metrics['RMSE (MW)'] - arima_metrics['RMSE (MW)']
    else:
        winner = 'PyTorch LSTM'
        margin = arima_metrics['RMSE (MW)'] - lstm_metrics['RMSE (MW)']
    
    print(f"\n🏆 Winner: {winner} (by {margin:.2f} MW RMSE)")

# Error statistics
print("\n📊 Error Statistics:")
for name, pred_series in [('ARIMA', arima_pred_series), ('LSTM', lstm_pred_series)]:
    errors = test.values - pred_series.values
    errors_clean = errors[~np.isnan(errors)]
    if len(errors_clean) > 0:
        print(f"\n{name}:")
        print(f"   Mean Error: {np.mean(errors_clean):.2f} MW")
        print(f"   Std Error: {np.std(errors_clean):.2f} MW")
        print(f"   Min Error: {np.min(errors_clean):.2f} MW")
        print(f"   Max Error: {np.max(errors_clean):.2f} MW")

## 11. Visualization

In [ ]:
# =============================================================================
# VISUALIZATION
# =============================================================================

# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Training Loss', linewidth=2)
axes[0].plot(val_losses, label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('LSTM Training History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Predictions visualization (last 7 days)
last_7_days = min(24 * 7, len(test))
actual_7d = test.iloc[-last_7_days:]
arima_7d = arima_pred_series.iloc[-last_7_days:]
lstm_7d = lstm_pred_series.iloc[-last_7_days:]

axes[1].plot(actual_7d.index, actual_7d.values, 
             label='Actual', color='#2c3e50', linewidth=2.5, alpha=0.9)
axes[1].plot(arima_7d.index, arima_7d.values, 
             label=f'ARIMA (RMSE: {arima_metrics["RMSE (MW)"]} MW)', 
             color='#e74c3c', linewidth=2, linestyle='--', alpha=0.8)
axes[1].plot(lstm_7d.index, lstm_7d.values, 
             label=f'PyTorch LSTM (RMSE: {lstm_metrics["RMSE (MW)"]} MW)', 
             color='#27ae60', linewidth=2, linestyle='-.', alpha=0.8)

axes[1].set_title('Last 7 Days of Predictions')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Demand (MW)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Add day markers
for date in pd.date_range(actual_7d.index.min(), actual_7d.index.max(), freq='D'):
    axes[1].axvline(x=date, color='gray', linestyle=':', alpha=0.3)

plt.suptitle('PyTorch ARIMA vs LSTM Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Error distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (name, pred_series, color) in enumerate([
    ('ARIMA', arima_pred_series, '#e74c3c'),
    ('PyTorch LSTM', lstm_pred_series, '#27ae60')
]):
    errors = test.values - pred_series.values
    errors_clean = errors[~np.isnan(errors)]
    
    if len(errors_clean) > 0:
        axes[idx].hist(errors_clean, bins=50, color=color, alpha=0.7, edgecolor='black')
        axes[idx].axvline(x=0, color='black', linestyle='--', linewidth=2)
        axes[idx].axvline(x=np.mean(errors_clean), color='blue', linestyle='-', linewidth=2,
                         label=f'Mean: {np.mean(errors_clean):.1f} MW')
        axes[idx].set_title(f'{name} Error Distribution')
        axes[idx].set_xlabel('Error (MW)')
        axes[idx].set_ylabel('Frequency')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Save Results and Model

In [ ]:
# =============================================================================
# SAVE RESULTS
# =============================================================================

print("💾 Saving results...\n")

# Save predictions
predictions_df = pd.DataFrame({
    'DateTime': test.index,
    'Actual': test.values,
    'ARIMA_Prediction': arima_pred_series.values,
    'LSTM_Prediction': lstm_pred_series.values,
    'ARIMA_Error': test.values - arima_pred_series.values,
    'LSTM_Error': test.values - lstm_pred_series.values
})

predictions_df.to_csv('pytorch_predictions.csv', index=False)
print("   ✅ Predictions saved to pytorch_predictions.csv")

# Save results
results_df.to_csv('pytorch_results.csv', index=False)
print("   ✅ Results saved to pytorch_results.csv")

# Save PyTorch model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'best_val_loss': best_val_loss,
    'hyperparameters': {
        'lookback': LOOKBACK,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs': len(train_losses),
        'hidden_size': model.hidden_size,
        'num_layers': model.num_layers
    }
}, 'pytorch_lstm_model.pth')
print("   ✅ Model saved to pytorch_lstm_model.pth")

# Save configuration
config = {
    'pytorch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda if torch.cuda.is_available() else None,
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'training_time': elapsed_time,
    'best_val_loss': best_val_loss,
    'final_metrics': {
        'arima': arima_metrics,
        'lstm': lstm_metrics
    }
}

with open('pytorch_config.json', 'w') as f:
    json.dump(config, f, indent=2, default=str)
print("   ✅ Configuration saved to pytorch_config.json")

print("\n" + "="*60)
print("✅ ANALYSIS COMPLETE!")
print("="*60)
print(f"\n📊 Final Summary:")
print(f"   Device: {device}")
print(f"   Training time: {elapsed_time:.1f} seconds")
print(f"   ARIMA RMSE: {arima_metrics['RMSE (MW)']} MW")
print(f"   LSTM RMSE: {lstm_metrics['RMSE (MW)']} MW")
if torch.cuda.is_available():
    print(f"   GPU Memory Used: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

# Final cleanup
clear_gpu_memory()